# Kiểm tra dữ liệu và thực nghiệm Seasonal Naive
**Học phần:** Phân tích dữ liệu kinh doanh — IS403.R11. **Nhóm thực hiện:** Nhóm 6.
Thực nghiệm kiểm tra cấu trúc dữ liệu M5 và đo thời gian dự báo trên 10 chuỗi sản phẩm–cửa hàng. Phương pháp Seasonal Naive sử dụng chu kỳ 7 ngày, lịch sử `d_1–d_1885` và chân trời dự báo 28 ngày.
Thời gian đo bao gồm thực thi hàm dự báo và kiểm tra đầu vào, không bao gồm đọc dữ liệu hoặc ghi tệp.


In [ ]:
import os, sys, subprocess, platform, json, zipfile, gc
from pathlib import Path
from datetime import datetime, timezone, timedelta

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    runtime, base = 'kaggle', Path('/kaggle/working')
elif 'COLAB_RELEASE_TAG' in os.environ:
    runtime, base = 'colab', Path('/content')
else:
    raise RuntimeError('Chạy notebook này trên Colab hoặc Kaggle; dùng lệnh README trên máy cá nhân.')
ROOT = base / 'IS403.R11-M5-Competition'
if not ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/haiphamt/IS403.R11-M5-Competition.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'PyYAML'], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))
print('Runtime:', runtime)


## Dữ liệu nghiên cứu
Bộ dữ liệu M5 gồm thông tin lịch và sự kiện (`calendar.csv`), giá bán (`sell_prices.csv`), hai bảng doanh số lịch sử (`sales_train_validation.csv`, `sales_train_evaluation.csv`) và mẫu định dạng dự báo (`sample_submission.csv`).
Mỗi dòng của bảng doanh số tương ứng với một cặp sản phẩm–cửa hàng; các cột `d_1`, `d_2`, … thể hiện số lượng bán theo ngày. Hai bảng doanh số có phần lịch sử trùng nhau và được đọc riêng.
Lần đo thời gian sử dụng 10 chuỗi đầu tiên trong `sales_train_validation.csv`. Kết quả này mô tả chi phí tính toán của phép dự báo, chưa dùng để đánh giá độ chính xác hoặc tính đại diện của mẫu.


In [ ]:
if runtime == 'colab':
    from google.colab import files
    names = ['calendar.csv', 'sell_prices.csv', 'sales_train_validation.csv', 'sales_train_evaluation.csv', 'sample_submission.csv']
    candidates = []
    for path in base.glob('*.zip'):
        if zipfile.is_zipfile(path):
            with zipfile.ZipFile(path) as archive:
                if set(names).issubset(archive.namelist()):
                    candidates.append(path)
    if not candidates:
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded]
        del uploaded
    if len(candidates) != 1:
        raise ValueError('Cần đúng một ZIP chứa đủ 5 file M5 gốc.')
    with zipfile.ZipFile(candidates[0]) as archive:
        if not set(names).issubset(archive.namelist()):
            raise ValueError('ZIP thiếu file M5 hoặc file CSV không nằm ngay ở thư mục gốc.')
        archive.extractall(ROOT / 'data' / 'raw', members=names)
    gc.collect()
else:
    print('Kaggle: đọc bộ M5 đã gắn trong Input.')


In [ ]:
import pandas as pd
import numpy as np
import yaml
from m5.data import load_raw_data

raw = load_raw_data()
shapes = pd.DataFrame([{'table': name, 'rows': len(table), 'columns': len(table.columns)} for name, table in raw.items()])
print(shapes.to_string(index=False))
out = ROOT / 'results' / 'tables'
out.mkdir(parents=True, exist_ok=True)
shapes.to_csv(out / f's05_data_shapes_{runtime}.csv', index=False)
environment = {
    'task': 'S05/S06', 'runtime': runtime, 'all_rows_read': True,
    'recorded_at': datetime.now(timezone(timedelta(hours=7))).isoformat(),
    'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'pyyaml': yaml.__version__,
    'os': platform.system(), 'logical_cpu_count': os.cpu_count(),
    'commit': subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip(),
    'source': 'https://www.kaggle.com/competitions/m5-forecasting-accuracy/data'
}
(out / f's05_environment_{runtime}.json').write_text(json.dumps(environment, indent=2) + '\n', encoding='utf-8')
print(json.dumps(environment, indent=2))
subprocess.run(['free', '-h'], check=True)
subprocess.run(['nproc'], check=True)
del raw
gc.collect()
subprocess.run([sys.executable, 'scripts/benchmark_seasonal_naive.py'], check=True)

RESULT_ZIP = base / f'is403_m5_checks_{runtime}.zip'
with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.glob(f'*_{runtime}.*')):
        archive.write(path, arcname=path.name)
print('Results:', RESULT_ZIP)


In [ ]:
if runtime == 'colab':
    files.download(str(RESULT_ZIP))
else:
    print('Tải ZIP kết quả từ Output của notebook:', RESULT_ZIP.name)
